# Titanic Survival Prediction — Gradient Boosting

An ML model with scikit-learn's histogram-based gradient boosting that predicts the
survival probability of a passenger.

**Before running:** put the Kaggle Titanic `train.csv` in the same folder as this notebook.

```bash
pip install scikit-learn pandas numpy matplotlib
```

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import (
    cross_val_score, StratifiedKFold, GridSearchCV, train_test_split
)
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)

## 2. Load data

In [ ]:
df = pd.read_csv("train.csv")
print(df.shape)
df.head()

In [ ]:
df.info()
df.isna().sum()

## 3. Feature engineering

- **Title** from the name (age / sex / social-status signal)
- **FamilySize** / **IsAlone** from `SibSp` + `Parch`
- **Deck** letter from `Cabin`
- Drop pure identifiers (`PassengerId`, `Name`, `Ticket`, `Cabin`, row index)

In [ ]:
def engineer(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["Title"] = (
        df["Name"].str.extract(r",\s*([^\.]+)\.", expand=False)
        .str.strip()
        .replace({
            "Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs",
            "Lady": "Rare", "Countess": "Rare", "Capt": "Rare",
            "Col": "Rare", "Don": "Rare", "Dr": "Rare",
            "Major": "Rare", "Rev": "Rare", "Sir": "Rare",
            "Jonkheer": "Rare", "Dona": "Rare",
        })
    )

    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    df["Deck"] = df["Cabin"].str[0].fillna("U")

    df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
    df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")],
                 errors="ignore")
    return df


data = engineer(df)
data.head()

In [ ]:
X = data.drop(columns=["Survived"])
y = data["Survived"]
X.head()

## 4. Preprocessing + model pipeline

In [ ]:
numeric_features = ["Age", "Fare", "SibSp", "Parch", "FamilySize", "IsAlone", "Pclass"]
categorical_features = ["Sex", "Embarked", "Title", "Deck"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_features),
    ]
)

model = Pipeline([
    ("prep", preprocess),
    ("clf", HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=400,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        early_stopping=True,
        random_state=RANDOM_STATE,
    )),
])
model

## 5. Cross-validation

Expect roughly **0.82 – 0.84** accuracy.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
print(f"CV accuracy: {scores.mean():.3f} +/- {scores.std():.3f}")
scores

## 6. Hyperparameter tuning (optional, slow)

In [ ]:
param_grid = {
    "clf__learning_rate": [0.03, 0.05, 0.1],
    "clf__max_iter": [300, 500],
    "clf__max_leaf_nodes": [15, 31, 63],
    "clf__min_samples_leaf": [10, 20, 30],
}

search = GridSearchCV(model, param_grid, cv=cv, scoring="roc_auc", n_jobs=-1)
search.fit(X, y)
print(search.best_params_)
print(f"best ROC AUC: {search.best_score_:.3f}")
model = search.best_estimator_

## 7. Fit final model and evaluate

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

model.fit(X_tr, y_tr)
proba = model.predict_proba(X_te)[:, 1]

print(classification_report(y_te, model.predict(X_te)))
print("ROC AUC:", round(roc_auc_score(y_te, proba), 3))

RocCurveDisplay.from_predictions(y_te, proba)
plt.show()

## 8. Feature importance (permutation)

In [ ]:
perm = permutation_importance(
    model, X_te, y_te, n_repeats=20, random_state=RANDOM_STATE, scoring="roc_auc"
)
importances = (
    pd.Series(perm.importances_mean, index=X.columns)
    .sort_values(ascending=True)
)
importances.plot.barh()
plt.xlabel("Drop in ROC AUC when shuffled")
plt.title("Permutation importance")
plt.show()
importances.sort_values(ascending=False)

## 9. Predict the survival probability of any person

Build a one-row frame with the **raw** columns, run it through `engineer()`, then the pipeline.

In [ ]:
def predict_person(pipeline, **fields) -> float:
    raw = pd.DataFrame([{
        "PassengerId": 0, "Survived": np.nan, "Pclass": 3,
        "Name": "Doe, Mr. John", "Sex": "male", "Age": 30.0,
        "SibSp": 0, "Parch": 0, "Ticket": "XXX", "Fare": 8.05,
        "Cabin": np.nan, "Embarked": "S",
        **fields,
    }])
    person = engineer(raw).drop(columns=["Survived"])
    return float(pipeline.predict_proba(person)[:, 1][0])


# 25-year-old woman, 1st class, paid GBP 100, embarked at Cherbourg
p1 = predict_person(
    model, Name="Smith, Mrs. Jane", Sex="female", Age=25, Pclass=1,
    Fare=100.0, Embarked="C", Cabin="C85",
)
print(f"Woman, 1st class:  {p1:.1%}")

# 40-year-old man, 3rd class
p2 = predict_person(model, Name="Brown, Mr. Bob", Sex="male", Age=40, Pclass=3, Fare=7.9)
print(f"Man, 3rd class:    {p2:.1%}")

## 10. Notes

- **Pipeline** ensures imputation / encoding are learned only on training folds — CV scores stay honest.
- `HistGradientBoostingClassifier` accepts NaN directly; the numeric `SimpleImputer` is kept only so you could swap in the classic `GradientBoostingClassifier`.
- Drop pure identifiers (row index `X`, `PassengerId`) — they only add noise / leakage.
- Swap the classifier for `xgboost`, `lightgbm`, or `catboost` (same sklearn API) for a possible extra point of accuracy.